In [ ]:
#!pip uninstall huetracer -y
#!pip install huetracer
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
import os
import random
import numpy as np
import pandas as pd
import scvi
import gc
import math
import bin2cell as b2c
import torch
from itertools import cycle
from sklearn.neighbors import NearestNeighbors
import importlib
import gdown
import zipfile
import adjustText as at
import scipy
import huetracer
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore", message=".*squidpy.*")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10,10],dpi=100)

scvi.settings.seed = 0
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else"mps" if torch.backends.mps.is_available() else "cpu")
device_str = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
### parameters to be input
SAMPLE_NAME = 'E16_15'
lib_id = SAMPLE_NAME # list(sp_adata.uns['spatial'].keys())[0]

path = os.path.expanduser("~")+"/Desktop/space/" + SAMPLE_NAME
save_path_for_today = os.path.expanduser("~")+"/tmp/outputs/250610_" + SAMPLE_NAME
visium_path = "/Volumes/Public/data/tendon_mouse/visium_HD/"
source_image_path = visium_path + "he/" + SAMPLE_NAME + ".tif"
expression_path = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_002um"
### optional 8/16 um binned dataset
expression_path_8um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_008um"
expression_path_16um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_016um"
###

# area to be analyzed
# ## GCTB spatial G1, FFPE9
mask_large_x1, mask_large_x2, mask_large_y1, mask_large_y2 = 450, 1950, 250, 1750

# Species = "Human"
Species = "Mouse"

# List of target gene names
if Species == "Human":
    target_genes = [
        'TNFSF11' # Add more genes here if needed
    ]
    prefix_mt = 'MT-'
else:
    target_genes = [
        'Col1a1' # Add more genes here if needed
    ]
    prefix_mt = 'mt-'

### ligand-receptor data obtained from nichenet, download only once
# # 1. Google Drive link URL
# url = "https://drive.google.com/uc?export=download&id=1pMpGUfsrDNWmZ_osfSglX5MrL8e8vOeA"
# # 2. File name
# output = "ligand_target_df.csv.zip"
# # 3. Download
# gdown.download(url, output, quiet=False)
# # 4. ZIP file extraction
# with zipfile.ZipFile(output, 'r') as zip_ref:
#     zip_ref.extractall("ligand_target_df")

Gene_to_analyze = "LIF"
# Gene_to_analyze = "CSF1"

# Definition of neighborhood cells
neighbor_cell_numbers = 19

#role = 'sender'
role = 'receiver'
each_display_num = 3

# Volcano plot of gene expression between clusters
group1_environments = ['0', '2'] # Microenvironmentのカテゴリ名。文字列で定義
group2_environments = ['1', '3'] # Microenvironmentのカテゴリ名。文字列で定義

# setting for filenames
label_image_filename = "he_labels_image.pdf"
h5ad_filename = SAMPLE_NAME + "_b2c.h5ad"
h5ad_full_filename = SAMPLE_NAME + "_2um.h5ad"
h5ad_predicted_full_filename = SAMPLE_NAME + "_nucleus_predicted.h5ad"
h5ad_sc_filtered_full_filename = SAMPLE_NAME + "_single_cell_filtered.h5ad"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
save_spatial_plot_path = os.path.join(save_path_for_today, "cropped_spatial_plot.svg")
save_svg_path = os.path.join(save_path_for_today, "spatial_salvage_labels.svg")
h5ad_save_path = os.path.join(save_path_for_today, h5ad_filename)
h5ad_full_save_path = os.path.join(save_path_for_today, h5ad_full_filename)
h5ad_predicted_full_save_path = os.path.join(save_path_for_today, h5ad_predicted_full_filename)
h5ad_sc_filtered_full_save_path = os.path.join(save_path_for_today, h5ad_sc_filtered_full_filename)
h5ad_microenvironment_full_save_path = os.path.join(save_path_for_today, h5ad_sc_microenvironment_full_filename)

os.chdir(path)
os.makedirs(save_path_for_today, exist_ok=True)

sp_adata_predicted = sc.read_h5ad(h5ad_predicted_full_save_path)
sp_adata_raw = sc.read_h5ad(h5ad_save_path)

target_cell_type = sp_adata_predicted.obs['predicted_cell_type'].value_counts().idxmax()
# target_cell_type = 'Airway epithelial cells (CAPN8+, ELF3+)' # 'Tumor', 'GiantCell', etc., replace with your desired CellType


In [ ]:
# Microenvironment estimation with variational autoencoder, gene expression data of 18 cells around the center cell was used.
# === Preprocessing ===
cell_mask = ((sp_adata_predicted.obs['array_row'] >= mask_large_x1) & 
             (sp_adata_predicted.obs['array_row'] <= mask_large_x2) & 
             (sp_adata_predicted.obs['array_col'] >= mask_large_y1) & 
             (sp_adata_predicted.obs['array_col'] <= mask_large_y2)
            )
sp_adata_microenvironment = sp_adata_predicted.copy()[cell_mask]
sp_adata_microenvironment.X = sp_adata_microenvironment.raw.X.copy()

# 1. Cell count by cell labels
group_counts = sp_adata_microenvironment.obs['scvi_predicted_labels'].value_counts()
valid_groups = group_counts[group_counts > 1].index.tolist()

# 2. Exclude cell types with only 1 cell count
filtered_adata = sp_adata_microenvironment.copy()[
    sp_adata_microenvironment.obs['scvi_predicted_labels'].isin(valid_groups)
]
sc.pp.normalize_total(filtered_adata, target_sum = 1e6)
sc.pp.log1p(filtered_adata)

filtered_adata.raw = None

# 3. Select genes with DEG analysis
sc.tl.rank_genes_groups(
    filtered_adata,
    # groupby='scvi_predicted_labels',
    groupby='predicted_cell_type',
    #groupby='leiden_nucleus',
    method='wilcoxon',
    n_genes=100
)

top_genes_df = pd.DataFrame(filtered_adata.uns['rank_genes_groups']['names'])
top_genes_list = top_genes_df.values.flatten().tolist()
top_genes_list = [g for g in top_genes_list if pd.notnull(g)]
common_hvg = list(set(top_genes_list))
common_hvg = [g for g in common_hvg if g in filtered_adata.var_names]

sc.pp.highly_variable_genes(filtered_adata, n_top_genes=100, flavor='seurat_v3')
ref_hvg_100 = filtered_adata.var[filtered_adata.var['highly_variable']].index.tolist()
all_genes = set(common_hvg) | \
            set(ref_hvg_100)
final_gene_list = [g for g in all_genes if g in sp_adata_microenvironment.var_names]

# sc.pp.normalize_total(sp_adata_microenvironment, target_sum = 1e0)
sp_adata_microenvironment = sp_adata_microenvironment[:, final_gene_list].copy()
coords = sp_adata_microenvironment.obs[["array_row", "array_col"]].values
X = sp_adata_microenvironment.X.toarray() if hasattr(sp_adata_microenvironment.X, "toarray") else sp_adata_microenvironment.X  # shape: (n_cells, n_genes)
bin_counts = sp_adata_microenvironment.obs['bin_count'].values.astype(float)
bin_counts[bin_counts == 0] = 1e-9 
X = X / bin_counts.reshape(-1, 1)
sp_adata.layers['binned_normalized'] = X
data_min = X.min()
data_max = X.max()
X = (X - data_min) / (data_max - data_min)
cell_types = sp_adata_microenvironment.obs['predicted_cell_type']
#cell_types = sp_adata_microenvironment.obs['cluster_cell_type']

print(f"Device: {device}")

# analysis
analyzer = huetracer.SpatialMicroenvironmentAnalyzer(coords, X, k_neighbors=neighbor_cell_numbers)
indices, microenv_data = analyzer.build_microenvironment_data()
vae_model = analyzer.train_vae(latent_dim=32, epochs=1000, batch_size=16384, lr=4e-4, dim_1 = 128, dim_2 = 128, weight_decay=1e-4, beta=1)
analyzer.extract_latent_features()
umap_embedding, clusters = analyzer.perform_umap_clustering(cell_type_data=cell_types)
# visualization
analyzer.visualize_results()
analyzer.visualize_scanpy_results()

huetracer.plot_all_clusters_highlights(analyzer)
huetracer.plot_all_cell_type_highlights(analyzer)

sp_adata_microenvironment.obs['predicted_microenvironment'] = analyzer.adata.obs['leiden'].values.astype(str)

selected_cells = sp_adata_microenvironment

# ---------- 2. hires座標計算 ----------
sf_hires = selected_cells.uns["spatial"][lib_id]["scalefactors"]["tissue_hires_scalef"]
xy = (
    pd.DataFrame(selected_cells.obsm["spatial"] * sf_hires, columns=["x", "y"], index=selected_cells.obs_names)
    .join(selected_cells.obs["object_id"])
    .reset_index()
    .rename(columns={"index": "cell_id"})
)
merged = xy.merge(selected_cells.obs, on="object_id", how="inner")
merged["predicted_microenvironment"] = clusters
merged["group"] = merged["predicted_microenvironment"]
merged["predicted_cell_type"] = sp_adata_microenvironment.obs["predicted_cell_type"].values.astype(str)
predicted_microenvironment_original = analyzer.adata.obs['leiden'].values.astype(str)
predicted_cell_type_original = sp_adata_microenvironment.obs["predicted_cell_type"].values.astype(str)
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')
sp_adata_microenvironment.obs['predicted_cell_type'] = sp_adata_microenvironment.obs['predicted_cell_type'].astype('category')
merged["predicted_microenvironment"] = merged["group"].astype('category')
# ---------- 5. 色マッピング ----------
hires_img = selected_cells.uns["spatial"][lib_id]["images"]["hires"]
h, w = hires_img.shape[:2]

### Optional: Change microenvironment names and cell type names of selected cells
Install jupyter lab extensions

- jupyter-matplotlib # Matplotlib Jupyter Interactive Widget
- jupyter-widgets-jupyterlab-manager # The JupyterLab extension providing Jupyter widgets.
- jupyterlab-plotly # The plotly Jupyter extension


How to use:
1. Select the microenvironments you want to display in Display Groups (multiple selections possible).
2. Select the selection method in Mode (Lasso/Rectangle).
3. Select the area with the mouse.
4. Apply Selection.
5. Zoom in/out with the Zoom slider.
6. Modify microenvironments and cell types as you wish.



In [ ]:
# Check microenvironment clusters and cell types before modification
# adata_filtered = sp_adata_microenvironment[~sp_adata_microenvironment.obs['predicted_microenvironment'].isna(), :].copy()
%matplotlib widget
plt.close('all')
sc.pl.spatial(
    sp_adata_microenvironment,
    color='predicted_microenvironment',
    title='Predicted microenvironment',
    size=20,alpha_img=0.2,
    img_key='hires',
    legend_fontsize=5,
    groups=None,
    spot_size=1,
    frameon=False
)
sc.pl.spatial(sp_adata_microenvironment, color='predicted_cell_type',
              title='Predicted cell type',
              size=20,
              alpha_img=0.2,
              img_key='hires', legend_fontsize=5,
              groups=None,
              spot_size=1,
              frameon=False)


In [ ]:
# Microenvironment modification
plt.close('all')
%matplotlib widget
huetracer.lasso_selection_microenvironment(sp_adata_microenvironment, merged, lib_id, clusters)

In [ ]:
# Cell type modification
plt.close('all')
%matplotlib widget
huetracer.lasso_selection_cell_type(sp_adata_microenvironment, merged, lib_id, clusters)

In [ ]:
# Check modification result
%matplotlib widget
plt.close('all')
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')

sc.pl.spatial(
    sp_adata_microenvironment,
    color='predicted_microenvironment',
    title='Predicted microenvironment',
    size=20,alpha_img=0.2,
    img_key='hires',
    legend_fontsize=5,
    groups=None,
    spot_size=1,
    frameon=False
)

sc.pl.spatial(sp_adata_microenvironment, color='predicted_cell_type',
              title='Predicted predicted_cell_type',
              size=20,
              alpha_img=0.2,
              img_key='hires', legend_fontsize=5,
              groups=None,
              spot_size=1,
              frameon=False)

In [ ]:
# Reset modification
# 
# sp_adata_microenvironment.obs['predicted_microenvironment'] = predicted_microenvironment_original
# sp_adata_microenvironment.obs['predicted_cell_type'] = predicted_cell_type_original
# merged["predicted_microenvironment"] = predicted_microenvironment_original
# merged["predicted_cell_type"] = predicted_cell_type_original

#### Option end

In [ ]:
# Gene expression and microenvironment
importlib.reload(huetracer.plot)
plt.close('all')
%matplotlib widget
#%matplotlib inline
huetracer.plot.create_spatial_widget(
    sp_adata_raw, 
    sp_adata_microenvironment
)

In [ ]:
# Gene expression difference among clusters
importlib.reload(huetracer.plot)
plt.close('all')
%matplotlib inline
deg_results_df = huetracer.plot.plot_deg_by_microenvironment(
    sp_adata_raw=sp_adata_raw,
    sp_adata_microenvironment=sp_adata_microenvironment,
    target_cell_type=target_cell_type,
    #mask_coords=mask,
    n_genes=8, # トップ8遺伝子を表示
    save=True,  # プロットをPDFで保存
    save_path_for_today=save_path_for_today
)

In [ ]:
importlib.reload(huetracer.plot)
plt.close('all')
%matplotlib inline
huetracer.plot.create_report_plots(
    merged=merged,
    sp_adata_microenvironment=sp_adata_microenvironment,
    hires_img=hires_img,
    w=w,
    h=h,
    sample_name=SAMPLE_NAME,
    save_path=save_path_for_today
)

In [ ]:
importlib.reload(huetracer.plot)
plt.close('all')
%matplotlib widget
huetracer.plot.interactive_gene_histogram(
    adata=sp_adata_raw,
    gene_list=sorted(sp_adata_raw.var.index.tolist())
)

In [ ]:
# Spatial Gene Expression Viewer
common_cells = sp_adata_microenvironment.obs_names.intersection(sp_adata_raw.obs_names)
sp_adata = sp_adata_raw[common_cells].copy()
sp_adata_raw.obs['redicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')
sp_adata_raw.obs['predicted_cell_type'] = sp_adata_microenvironment.obs['predicted_cell_type'].astype('category')
viewer = huetracer.SpatialGeneExpressionViewer(
    sp_adata_microenvironment, 
    lib_id, 
    save_path=save_path_for_today
)
viewer.display()

In [ ]:
# Interactive Volcano Plot
importlib.reload(huetracer.widget)
importlib.reload(huetracer.statistics)
importlib.reload(huetracer.plot)
importlib.reload(huetracer)
common_cells = sp_adata_microenvironment.obs_names.intersection(sp_adata_raw.obs_names)
sp_adata = sp_adata_raw[common_cells].copy()

sp_adata_raw.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype('category')
sp_adata_raw.obs['predicted_cell_type'] = sp_adata_microenvironment.obs['predicted_cell_type'].astype('category')
plotter = huetracer.VolcanoPlotter(
    sp_adata_microenvironment,
    save_path=save_path_for_today
)
plotter.display()

In [ ]:
# Save spatial data for cell-cell interaction analysis 
sp_adata_microenvironment.write_h5ad(h5ad_microenvironment_full_save_path)